In [2]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [72]:
data = yf.download(
    "AAPL",
    start="2010-01-01",
    interval="1d"
)

[*********************100%***********************]  1 of 1 completed


In [73]:
import numpy as np

window_size = 50
future_days = 10

X = []
y = []

for start in range(len(close_prices) - window_size - future_days + 1):
    end = start + window_size

    # The 50 days that the model gets to see
    window = close_prices.iloc[start:end]

    # Convert absolute prices into relative movement
    normalized_window = (window / window.iloc[0]) - 1

    # Last price visible to the model
    current_price = close_prices.iloc[end - 1]

    # The NEXT 10 days
    future_window = close_prices.iloc[end:end + future_days]

    # Highest price reached during those 10 days
    max_future_price = future_window.max()

    # Best possible return during those 10 days
    max_return = max_future_price / current_price - 1

    X.append(normalized_window.values)
    y.append(max_return)

X = np.array(X)
y = np.array(y)

print("X:", X.shape)
print("y:", y.shape)

X: (4140, 50)
y: (4140,)


In [77]:
def build_dataset(close_prices, window_size=50, future_days=10):
    X = []

    max_returns = []
    days_to_max = []

    window_end_dates = []
    max_dates = []

    number_of_examples = (len(close_prices) - window_size - future_days + 1)

    for start in range(number_of_examples):
        end = start + window_size

        # -------------------------
        # PAST - visible to model
        # -------------------------

        window = close_prices.iloc[start:end]

        #normalize the values
        normalized_window = (
            (window / window.iloc[0]) - 1
        )

        # -------------------------
        # FUTURE - hidden from model
        # -------------------------

        future_window = close_prices.iloc[end:end + future_days]

        #normalize the returns as well
        future_returns = (
            (future_window / window.iloc[-1]) - 1
        )

        # Position & value of the highest return
        max_position = np.argmax(future_returns.values)

        max_value_return = future_returns.iloc[max_position]

        # trading day after entry"
        days_until_max = max_position + 1

        # -------------------------
        # Save this example
        # -------------------------

        X.append(normalized_window.values)

        max_returns.append(max_value_return)
        days_to_max.append(days_until_max)

        window_end_dates.append(window.index[-1])
        max_dates.append(
            future_window.index[max_position]
        )

    X = np.array(X)

    targets = pd.DataFrame({
        "window_end_date": window_end_dates,
        "max_value_return": max_returns,
        "days_to_max": days_to_max,
        "max_date": max_dates
    })

    return X, targets

In [ ]:
window_size=365
future_days=60
close_prices = data["Close"].squeeze()


X, targets = build_dataset(
    close_prices,
    window_size,
    future_days
)

print(targets[:10])

train_end = int(0.7 * len(X))
validation_end = int(0.85 * len(X))
gap_size = int(window_size + future_days)

# traning_set 70% of the data
X_train = X[:train_end]

# validation_set 15% of the data
X_validation = X[train_end + gap_size: validation_end]

# test_set 15% of the data
X_test = X[validation_end + gap_size:]

  window_end_date  max_value_return  days_to_max   max_date
0      2011-06-14          0.213482           29 2011-07-26
1      2011-06-15          0.234613           28 2011-07-26
2      2011-06-16          0.240651           27 2011-07-26
3      2011-06-17          0.259633           26 2011-07-26
4      2011-06-20          0.279367           25 2011-07-26
5      2011-06-21          0.240117           24 2011-07-26
6      2011-06-22          0.250457           23 2011-07-26
7      2011-06-23          0.242731           60 2011-09-19
8      2011-06-24          0.266891           60 2011-09-20
9      2011-06-27          0.245181           59 2011-09-20
